In [ ]:
from pathlib import Path

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

import mistsim.plotting as msplt

%matplotlib widget

In [ ]:
results_dir = Path("results")

# 7 nominal runs in order of increasing coverage
runs = [
    "mars",
    "mars-nv",
    "mars-lake",
    "mars-nv-lake",
    "mars-nv-lake-alma",
    "all-nominal",
    "alma-torres",
]
labels = [
    "MARS",
    "MARS+NV",
    "MARS+Lake",
    "MARS+NV+Lake",
    "MARS+NV+Lake+ALMA",
    "All nominal",
    "ALMA+Torres",
]

data = {}
x_true = None
lmax = None

for run, lab in zip(runs, labels):
    d = np.load(results_dir / f"{run}.npz")
    data[run] = d
    if x_true is None:
        x_true = d["x_true"]
        lmax = int(d["lmax"])

print(f"lmax = {lmax}, {len(data)} runs loaded")
for lab, run in zip(labels, runs):
    nvec = int(data[run]["nvec"])
    nsv = len(data[run]["Sigma"])
    print(f"  {lab}: {run} (nvec={nvec}, k={nsv})")

# Nominal Runs Comparison

Comparing 7 beam/site combinations for mapmaking performance.

## Singular Value Spectra

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for run, lab in zip(runs, labels):
    Sigma = data[run]["Sigma"]
    nvec = int(data[run]["nvec"])
    ax.semilogy(Sigma, lw=1.5, label=f"{lab} (nvec={nvec})")
ax.set_xlabel("Index", fontsize=14)
ax.set_ylabel("Singular value", fontsize=14)
ax.set_title("Singular Value Spectra")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.show()

## Wiener Filter Factors

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for run, lab in zip(runs, labels):
    Sigma = data[run]["Sigma"]
    nvec = int(data[run]["nvec"])
    Dnum = Sigma[:nvec]
    D = Dnum / (1 + Dnum**2)
    ax.plot(D, lw=1.5, label=lab)
ax.set_xlabel("Mode index", fontsize=14)
ax.set_ylabel(r"$D_i = \sigma_i / (1 + \sigma_i^2)$", fontsize=14)
ax.set_title("Wiener Filter Factors")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.show()

## Information Metrics

| Metric | Formula | Meaning |
|--------|---------|---------|
| Shannon info | $\frac{1}{2}\sum \log(1 + \sigma_i^2)$ | Total mutual information (nats) |
| Effective DoF | $\sum \sigma_i^2 / (1 + \sigma_i^2)$ | Number of well-constrained modes |
| Effective rank | $\#\{i : \sigma_i > 1\}$ | Modes with SNR > 1 |

In [ ]:
rows = []
for run, lab in zip(runs, labels):
    Sigma = data[run]["Sigma"]
    nvec = int(data[run]["nvec"])
    s2 = Sigma[:nvec] ** 2
    shannon = 0.5 * np.sum(np.log(1 + s2))
    eff_dof = np.sum(s2 / (1 + s2))
    eff_rank = int(np.sum(Sigma[:nvec] > 1))
    rows.append((lab, nvec, f"{shannon:.1f}", f"{eff_dof:.1f}", eff_rank))

header = f"{'Run':<22s} {'nvec':>6s} {'Shannon':>10s} {'Eff DoF':>10s} {'Eff rank':>10s}"
print(header)
print("-" * len(header))
for lab, nv, sh, dof, rk in rows:
    print(f"{lab:<22s} {nv:>6d} {sh:>10s} {dof:>10s} {rk:>10d}")

## Transfer Functions

In [ ]:
cl_true = hp.alm2cl(x_true)
valid = cl_true > 0

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for run, lab in zip(runs, labels):
    x_rec = data[run]["x_rec"]
    cl_cross = hp.alm2cl(x_true, x_rec)
    tf = np.zeros_like(cl_true)
    tf[valid] = cl_cross[valid] / cl_true[valid]
    ax.plot(np.arange(len(tf)), tf, lw=1.5, label=lab)
ax.axhline(1, color="k", ls="--", alpha=0.5)
ax.set_xlabel(r"$\ell$", fontsize=14)
ax.set_ylabel(r"$C_\ell^{\rm cross} / C_\ell^{\rm true}$", fontsize=14)
ax.set_title("Transfer Function Comparison")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.show()

## Power Spectra

In [ ]:
ell = np.arange(len(cl_true))

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
ax.plot(ell, cl_true, "k-", lw=2.5, label="True")
for run, lab in zip(runs, labels):
    x_rec = data[run]["x_rec"]
    ax.plot(ell, hp.alm2cl(x_rec), lw=1.5, label=lab)
ax.set_yscale("log")
ax.set_xlabel(r"$\ell$", fontsize=14)
ax.set_ylabel(r"$C_\ell$", fontsize=14)
ax.set_title("Power Spectra Comparison")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(left=1)
plt.show()

## Cumulative Residual RMS vs ell

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for run, lab in zip(runs, labels):
    x_rec = data[run]["x_rec"]
    cl_res = hp.alm2cl(x_rec - x_true)
    ell_arr = np.arange(len(cl_res))
    weights = (2 * ell_arr + 1) / (4 * np.pi)
    rms_cum = np.sqrt(np.cumsum(weights * cl_res))
    ax.semilogy(ell_arr[1:], rms_cum[1:], lw=1.5, label=lab)
ax.set_xlabel(r"$\ell_{\rm max}$", fontsize=14)
ax.set_ylabel("Cumulative residual RMS [K]", fontsize=14)
ax.set_title("RMS of (recovered - true) vs included modes")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.show()

## Map Comparison Grid

In [ ]:
x_recs = [data[run]["x_rec"] for run in runs]

fig = msplt.plot_comparison_grid(
    x_true, x_recs, labels, lmax, plot_lmax=10,
    nside=128, plot_galactic=True, frac_range=1.0,
    ratio=True, orientation="vertical",
)
plt.show()